In [30]:
import zarr
import xarray as xr
import matplotlib.pyplot as plt
from pathlib import Path
import pandas as pd

import dask.array as da
import dask.dataframe as dd

import datashader as ds
import colorcet as cc
import pvdeg

In [2]:
live_data_copy = Path("/projects/inspire/PySAM-MAPS/v1.2/")

data_dir = live_data_copy / "final-backup/"
gid_file = live_data_copy / "gid-lat-lon.csv"

In [3]:
gids_mapping_df = pd.read_csv(gid_file, index_col=0)
gids_mapping_df.index.name = "gid"

#gids_mapping_df = ddf.from_pandas(gids_mapping_df) # might not be worth it to do this 
gids_mapping_df

,latitude,longitude
gid,,
0,-15.95,-179.98
1,-15.99,-179.98
2,-16.03,-179.98
3,-16.07,-179.98
4,-16.11,-179.98
...,...,...
2018262,16.01,-22.50
2018263,15.97,-22.50
2018264,15.93,-22.50


In [4]:
confs = sorted(list(data_dir.iterdir()))

In [5]:
conf_data = xr.open_zarr(confs[0])

In [45]:
mean_subset = pvdeg.utilities.gids_dataset_to_coords_dataset(
    conf_data.edgetoedge.mean(dim="time"),
    gids_mapping_df
)

# lon2d, lat2d = xr.broadcast(mean_subset.longitude, mean_subset.latitude)
# broadcast in the order of da dims, then force exact dim order match
lat2d, lon2d = xr.broadcast(da.latitude, da.longitude)
lon2d = lon2d.transpose(*da.dims)
lat2d = lat2d.transpose(*da.dims)

da = mean_subset.assign_coords(
    longitude_2d=lon2d,
    latitude_2d=lat2d,
)

/kfs3/scratch/tford/envs/render/lib/python3.13/site-packages/pvdeg/utilities.py:1567: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  stacked = ds_gids.drop(["gid"])


In [46]:
import hvplot.xarray  # noqa
import cartopy.crs as ccrs

plot = da.hvplot.quadmesh(
    x="longitude_2d",
    y="latitude_2d",
    z="edgetoedge",
    geo=True,
    crs=ccrs.PlateCarree(),
    projection=ccrs.AlbersEqualArea(
        central_longitude=-96,
        central_latitude=37.5,
        standard_parallels=(29.5, 45.5),
    ),
    project=True,
    rasterize=True,
    coastline=True,
    cmap="inferno",
    colorbar=True,
    width=2000,
    height=1200,
)

plot

:DynamicMap   []
   :Overlay
      .Image.I     :Image   [longitude_2d,latitude_2d]   (edgetoedge)
      .Coastline.I :Feature   [Longitude,Latitude]

In [61]:
# Source - https://stackoverflow.com/a/65267737
# Posted by Daniel Edler
# Retrieved 2026-03-15, License - CC BY-SA 4.0

import holoviews as hv
from bokeh.io import export_svgs

def export_svg(obj, filename):
    plot_state = hv.renderer('bokeh').get_plot(obj).state
    plot_state.output_backend = 'svg'
    export_svgs(plot_state, filename=filename)

export_svg(plot, 'conf0-inferno-fullres.svg')


RuntimeError: Neither firefox and geckodriver nor a variant of chromium browser and chromedriver are available on system PATH. You can install the former with 'conda install -c conda-forge firefox geckodriver'.

In [60]:
import holoviews as hv
hv.save(plot, "conf0-inferno-fullres.png")

RuntimeError: Neither firefox and geckodriver nor a variant of chromium browser and chromedriver are available on system PATH. You can install the former with 'conda install -c conda-forge firefox geckodriver'.